In [12]:
# STEP ONE this step involves finding the data to study, ensure its from a reliable source. 

In [13]:
# STEP TWO document the data, what is it about? its features, sources and other legal info such as licenses. 

In [14]:
import pandas as pd
import numpy as np
df = pd.read_csv('https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv')
df.head()
#this block loads the data set from github and df.head displays the first 5 rows. 
#  STEP THREE LOAD THE DATA

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [15]:
print("Records:", df.shape[0])   # shape[0] = number of rows
print("Features:", df.shape[1])  # shape[1] = number of columns
df.dtypes                         # data type of every column (int, float, object, etc.)
#this chunk prints the features of the data table, shape/size, data table etc. 
# STEP FOUR EXPLORE THE DATA AND ITS FEATURES


Records: 344
Features: 8


species                  str
island                   str
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                      str
year                   int64
dtype: object

In [16]:
print("--- Missing values ---")
print(df.isna().sum())                     # blanks per column
 
print("--- Duplicate rows ---")
print(df.duplicated().sum())               # exact copies of a row
 
print("--- Unique species ---")
print(df["species"].unique())              # distinct species spellings
 
print("--- Island counts ---")
print(df["island"].value_counts())         # rows per island
 
print("--- Summary statistics: measurements ---")
continuous_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
print(df[continuous_cols].describe())      # stats, real numbers only
 
print("--- Year counts ---")
print(df["year"].value_counts().sort_index())  # rows per year, not an average
 
print("--- Summary statistics: categories ---")
print(df.describe(include="object"))       # stats for text columns

# STEP FIVE CLEAN, INSPECT AND TRANSFORM THE DATA. 


--- Missing values ---
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64
--- Duplicate rows ---
0
--- Unique species ---
<StringArray>
['Adelie', 'Gentoo', 'Chinstrap']
Length: 3, dtype: str
--- Island counts ---
island
Biscoe       168
Dream        124
Torgersen     52
Name: count, dtype: int64
--- Summary statistics: measurements ---
       bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g
count      342.000000     342.000000         342.000000   342.000000
mean        43.921930      17.151170         200.915205  4201.754386
std          5.459584       1.974793          14.061714   801.954536
min         32.100000      13.100000         172.000000  2700.000000
25%         39.225000      15.600000         190.000000  3550.000000
50%         44.450000      17.300000         197.000000  4050.000000
75%         48.500000   

C:\Users\chris\AppData\Local\Temp\ipykernel_19008\142300114.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include="object"))       # stats for text columns


In [17]:
# STEP SIX what does the data tell you? missing rows? columns? duplicates? 
# if the above conditions do appear, can you create a reasonable assumption to explain WHY those conditions occured?
# in ths case, sex was missing in 11 insances. perhaps 11 times it was unsafe to collect this data. 2 instances of missing island data, 
# perhaps the penguin was not on an island at the time of collection. 
# 2 instances of missing body mass, perhaps the penguin was too young to weigh.
# all these asumptions help fortify the data and give it meaning. 

In [ ]:
# STEP SEVEN cluster the data without labels. 
from sklearn.preprocessing import StandardScaler  # rescales features to comparable ranges
from sklearn.cluster import KMeans                 # groups similar rows together
 
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
clean = df.dropna(subset=features)                 # drop rows missing any of these
#this commands ensures that only clean data with no NaN values is used. this avoids skewing the data. 
 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(clean[features])   # rescale so no feature dominates
#cleans features, so that big or small numbers dont dominate the clustering. this is normalisation of the data. 
 
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10) #n_init = 10 means the algorithm will run 10 times with 
# different initializations and return the best output.
clean["cluster"] = kmeans.fit_predict(X_scaled)     # assign each row to a cluster
 
pd.crosstab(clean["cluster"], clean["species"])     # compare clusters to the real species


species,Adelie,Chinstrap,Gentoo
cluster,,,
0,24,63,0
1,0,0,123
2,127,5,0


In [ ]:
# STEP EIGHT Predict and check accuracy of the model.
from sklearn.model_selection import train_test_split  # splits data into train/test sets
from sklearn.tree import DecisionTreeClassifier         # the model we'll train
from sklearn.metrics import accuracy_score               # scores how many predictions were correct
 
X = clean[features]        # inputs (the measurements)
y = clean["species"]       # the label we're trying to predict
 
X_train, X_test, y_train, y_test = train_test_split (
    X, y, test_size=0.3, random_state=42, stratify=y
)
# 70% train, 30% test, keep species proportions balanced
 
model = DecisionTreeClassifier(max_depth=4, random_state=42) #decision tree model is used to predict the species of penguins based on their measurements. 
# max_depth=4 means the tree can have a maximum of 4 levels, which helps prevent overfitting.
model.fit(X_train, y_train)   # learn patterns from the training data
 
predictions = model.predict(X_test)   # guess species on data it hasn't seen
print("Accuracy:", round(accuracy_score(y_test, predictions) * 100, 2), "%")


Accuracy: 99.03 %
